In [1]:
exec(open(__import__('pathlib').Path(__vsc_ipynb_file__).parent.parent / 'src' / 'display_html.py').read())

## L_CA1 — CA1 Field

**Role**: Convergence of MSP (ECin → CA1, slow) and TSP (CA3 → CA1, fast).
The plus-phase ECout back-projection corrects CA1 toward the target item.

**Key parameters (Schapiro 2017 §2.a.iv)**:
- `W_ECin`: MSP pathway — fully connected; lr=0.05 (slow → community statistics)
- `W_CA3`: TSP pathway — fully connected; lr=0.4 (fast → episodic binding)
- `W_ECout`: back-projection — active only in plus phase (Q4)
- `k_frac = 0.10`: ~10% active (less sparse than DG; §2.a.iv)

**Phase structure (Schapiro 2017 §2.b)**:
- Q1 (cycles 1–25): pass `a_CA3=zeros` → ECin-dominant (theta trough; encoding)
- Q2–Q3 (cycles 26–75): pass `a_ECin=zeros` → CA3-dominant (theta peak; retrieval)
- Q4 (cycles 76–100): pass `a_ECout=target` → plus phase (correction)

**Understanding check**: Why does the learning rate for W_ECin need to be 10× slower than W_CA3?  
→ MSP must accumulate statistics across many trials to reflect community structure.  
If lr_MSP were as fast as lr_TSP, each trial would overwrite previous statistical patterns.  
TSP needs one-shot binding; if too slow, the episode is lost before W_CA3 strengthens.

In [ ]:
import sys
from pathlib import Path

SRC = str((Path(__vsc_ipynb_file__).parent.parent / 'src').resolve())
sys.path.insert(0, SRC)

import torch
from layer import L_DG, L_CA3, L_CA1

torch.manual_seed(42)
dg = L_DG(n_input=15, n_DG=100, k_frac=0.01, ecin_frac=0.25)
dg.W.data = torch.randn(15, 100) * 0.1
a_A = torch.zeros(15); a_A[0] = 1.0; a_A[1] = 0.9
dg.reset(); act_A = dg(a_A).clone()

torch.manual_seed(0)
ca3 = L_CA3(n_DG=100, n_CA3=50, k_frac=0.10, dg_frac=0.05)
ca3.W_ff.data  = torch.randn(100, 50) * 0.1
ca3.W_rec.data = torch.randn(50, 50)  * 0.05
ca3.reset(); act_ca3 = ca3(act_A).clone()

In [ ]:
# Instantiate and inspect structure
ca1 = L_CA1(n_items=15, n_CA3=50, n_CA1=50, k_frac=0.10, use_euler=True)

print(f"W_ECin  shape: {ca1.W_ECin.shape}")    # (15, 50)  MSP
print(f"W_CA3   shape: {ca1.W_CA3.shape}")     # (50, 50)  TSP
print(f"W_ECout shape: {ca1.W_ECout.shape}")   # (15, 50)  back-projection
print(f"n_active target: {max(1, int(ca1.k_frac * ca1.n_CA1))}")  # 5 units

In [ ]:
# Phase switch: Q1 (ECin-dominant) vs Q2-Q3 (CA3-dominant)
torch.manual_seed(7)
ca1.W_ECin.data = torch.randn(15, 50) * 0.1
ca1.W_CA3.data  = torch.randn(50, 50) * 0.1

# Q1: ECin active, CA3 zeroed
ca1.reset()
act_q1 = ca1(a_A, torch.zeros(50)).clone()

# Q2-Q3: ECin zeroed, CA3 active
ca1.reset()
act_q23 = ca1(torch.zeros(15), act_ca3).clone()

cos = torch.nn.functional.cosine_similarity(act_q1.unsqueeze(0), act_q23.unsqueeze(0)).item()
print(f"Q1  active: {(act_q1  > 0).sum().item()} / {ca1.n_CA1}")
print(f"Q23 active: {(act_q23 > 0).sum().item()} / {ca1.n_CA1}")
print(f"Cosine(Q1, Q2-3): {cos:.3f}  (different pathways → different CA1 patterns)")

In [ ]:
# Plus phase: adding a_ECout should shift CA1 activity toward ECout pattern
a_ecout_target = torch.zeros(15); a_ecout_target[3] = 1.0   # next item = item 3
ca1.W_ECout.data = torch.randn(15, 50) * 0.1

ca1.reset()
act_minus_end = ca1(a_A, act_ca3).clone()                              # Q2-Q3 final
act_plus      = ca1(a_A, act_ca3, a_ecout=a_ecout_target).clone()     # Q4

cos_mp = torch.nn.functional.cosine_similarity(
    act_minus_end.unsqueeze(0), act_plus.unsqueeze(0)
).item()
print(f"Cosine(ActM, ActP): {cos_mp:.3f}")
print("ActP should differ from ActM: ECout back-projection shifts CA1 toward target.")

In [ ]:
# CHL weight update — verify lr ratio W_CA3 / W_ECin ≈ 8× (lr_TSP/lr_MSP = 0.4/0.05)
ca1_test = L_CA1(n_items=15, n_CA3=50, n_CA1=50, k_frac=0.10, lr_MSP=0.05, lr_TSP=0.4)
torch.manual_seed(2)
ca1_test.W_ECin.data  = torch.randn(15, 50) * 0.1
ca1_test.W_CA3.data   = torch.randn(50, 50) * 0.1
ca1_test.W_ECout.data = torch.randn(15, 50) * 0.1

ca1_test.reset()
a_ca1_m = ca1_test(a_A, act_ca3).clone()
a_ca1_p = ca1_test(a_A, act_ca3, a_ecout=a_ecout_target).clone()

W_ECin_before  = ca1_test.W_ECin.data.clone()
W_CA3_before   = ca1_test.W_CA3.data.clone()
W_ECout_before = ca1_test.W_ECout.data.clone()

ca1_test.update_weights(
    a_ECin=a_A,
    a_CA3_minus=act_ca3,  a_CA3_plus=act_ca3,
    a_ECout_plus=a_ecout_target,
    a_CA1_minus=a_ca1_m,  a_CA1_plus=a_ca1_p,
)

dW_ECin  = (ca1_test.W_ECin.data  - W_ECin_before).abs().max().item()
dW_CA3   = (ca1_test.W_CA3.data   - W_CA3_before).abs().max().item()
dW_ECout = (ca1_test.W_ECout.data - W_ECout_before).abs().max().item()

print(f"max |ΔW_ECin | = {dW_ECin:.5f}  (lr_MSP={ca1_test.lr_MSP})")
print(f"max |ΔW_CA3  | = {dW_CA3:.5f}  (lr_TSP={ca1_test.lr_TSP})")
print(f"max |ΔW_ECout| = {dW_ECout:.5f}  (lr_MSP={ca1_test.lr_MSP})")
print(f"Ratio ΔW_CA3 / ΔW_ECin ≈ {dW_CA3 / (dW_ECin + 1e-9):.1f}  (expect ~{ca1_test.lr_TSP/ca1_test.lr_MSP:.0f}×)")